In [0]:
from pyspark.sql.functions import col, current_timestamp

LANDING = "/Volumes/workspace/coffee/landing/online_orders"
CHECKPOINT = "/Volumes/workspace/coffee/landing/_checkpoints/bronze_online_orders"

bronze_stream = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", CHECKPOINT)
    .option("cloudFiles.inferColumnTypes", "true")
    .load(LANDING)
    .withColumn("source_file", col("_metadata.file_path"))
    .withColumn("ingested_at", current_timestamp()))

query = (bronze_stream.writeStream
    .option("checkpointLocation", CHECKPOINT)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable("workspace.coffee.bronze_online_orders"))

query.awaitTermination()

In [0]:
print(spark.table("workspace.coffee.bronze_online_orders").count())

In [0]:
%sql
SELECT order_id, customer_id, city, channel, _rescued_data
FROM workspace.coffee.bronze_online_orders
ORDER BY order_id;